# Notebook 03: Model development and interpretability

Derived from **`mmr comprehensive.ipynb`**. This notebook contains the **time-based train/test split**, **preprocessing pipelines** (median imputation; scaling for models that need it), **multiple regressors** (linear, ridge, lasso, SVR, KNN, MLP, random forest, gradient boosting), **performance metrics**, **prediction diagnostics** (scatter, residuals), **partial dependence**, and **permutation importance** for the selected best model.

**Prerequisite:** Notebook 01 (`country_year_modeling_panel_cleaned.csv`).

Data path: `../data/processed/country_year_modeling_panel_cleaned.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay


In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/country_year_modeling_panel_cleaned.csv")
df.head()


## Outcome and predictors (same definitions as original notebook)


In [ ]:
# Ensure year is int and iso3c clean
df["year"] = df["year"].astype(int)
df["iso3c"] = df["iso3c"].astype(str).str.strip().str.upper()

#because of the skewness, applied a log transformation to stabilize variance and improve model performance
#log_mmr = log(1 + MMR)
# If log_mmr isn't in the file, create it
if "log_mmr" not in df.columns:
    df["log_mmr"] = np.log1p(df["mmr"])

target = "log_mmr"

features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days",
]

# Keep only features that exist
features = [c for c in features if c in df.columns]
print("Using features:", features)


In [ ]:
#included in model: gdp per capita, health expenditure, fertility rate, skilled birth attendance, pm2.5 exposure(pollution), extreme heat dates, and cooling degree days
#variables represent economic, health systems, demographic, and climate drivers on MM
features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days"
]

features = [f for f in features if f in df.columns]


## Train/test split (no future leakage)


In [ ]:
#to prevent future data leakage and mimic real-world forecasting
#Training data: Years ≤ 2018
#Test data: Years > 2018

split_year = 2018

train = df[df["year"] <= split_year].copy()
test  = df[df["year"] > split_year].copy()

X_train = train[features]
y_train = train[target]

X_test  = test[features]
y_test  = test[target]

print("Train:", X_train.shape, " Test:", X_test.shape)


In [ ]:
numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])


In [ ]:
models = {
    #linear:
    "Linear Regression": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", Ridge(alpha=1.0))
    ]),
    "Lasso": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", Lasso(alpha=0.01))
    ]),
    #kernel based
    "SVR (RBF)": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", SVR(kernel="rbf", C=10, epsilon=0.1))
    ]),
    #distance based
    "KNN": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", KNeighborsRegressor(n_neighbors=10))
    ]),
    #neural network
    "MLP Regressor": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", MLPRegressor(hidden_layer_sizes=(64,32), max_iter=1000, random_state=42))
    ]),
    #tree based
    "Random Forest": Pipeline([
        ("prep", numeric_transformer_unscaled), 
        ("model", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", numeric_transformer_unscaled),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),
}


In [ ]:
# evaluate models using MAE, RMSE, R2
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    preds_log = model.predict(X_test)

    preds = np.expm1(preds_log)
    true = np.expm1(y_test)

    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(y_test, preds_log)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 (log-scale)": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


In [ ]:
#best model is SVR and indicates the importance of nonlinear relationships
#fertility, pollutions, and economic effects are probably non linear


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_model.fit(X_train, y_train)

print("Best model:", best_name)


## Predicted vs. actual (titles use the selected `best_name`)


In [ ]:
#scatter plot compares predicted MMR vs actual MMR. The dashed diagonal line represents perfect predictions. Points close to the line indicate better model performance.


In [ ]:
#strong clustering at low MMR means model performs well for low-risk countries
#wider spread at high MMR means model struggles with extreme values
#slight underprediction at very high MMR levels


In [ ]:
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

plt.figure(figsize=(7,6))
plt.scatter(y_true, y_pred, alpha=0.5)
plt.plot([y_true.min(), y_true.max()],
         [y_true.min(), y_true.max()],
         linestyle="--")

plt.xlabel("Actual MMR")
plt.ylabel("Predicted MMR")
plt.title(f"{best_name}: Predicted vs Actual (MMR Scale)")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_true, y_pred,
    s=18, alpha=0.25, edgecolors="none"
)

lo = min(y_true.min(), y_pred.min())
hi = max(y_true.max(), y_pred.max())
plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)

plt.xscale("symlog", linthresh=10)   # linear-ish near 0, log after 10
plt.yscale("symlog", linthresh=10)

plt.xlabel("Actual MMR (deaths per 100,000 live births)")
plt.ylabel("Predicted MMR (deaths per 100,000 live births)")
plt.title(f"{best_name}: Predicted vs Actual (symlog scale to reveal low-MMR structure)")
plt.grid(True, which="both", linewidth=0.4, alpha=0.4)
plt.show()


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(y_true, y_pred, s=18, alpha=0.25, edgecolors="none")

lo = min(y_true.min(), y_pred.min())
hi = max(y_true.max(), y_pred.max())
ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)

ax.set_xlabel("Actual MMR (deaths per 100,000 live births)")
ax.set_ylabel("Predicted MMR (deaths per 100,000 live births)")
ax.set_title(f"{best_name}: Predicted vs Actual (with zoomed low-MMR inset)")

# Inset zoom (adjust these limits if you want)
axins = inset_axes(ax, width="45%", height="45%", loc="upper left", borderpad=1)
axins.scatter(y_true, y_pred, s=12, alpha=0.25, edgecolors="none")
axins.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)

axins.set_xlim(0, 250)
axins.set_ylim(0, 250)
axins.set_title("Zoom: 0–250", fontsize=9)

mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4")
plt.show()


## Residual analysis


In [ ]:
#residual analysis
#slight funnel shape meaning heteroskedasticity
#larger variance at high predicted MMR
#model struggles more with extreme maternal mortality


In [ ]:
residuals = y_true - y_pred

plt.figure(figsize=(7,5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted MMR")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"{best_name} Residuals")
plt.show()


In [ ]:
residuals = y_true - y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals,
    s=18,
    alpha=0.25,
    edgecolors="none"
)

plt.axhline(0, linestyle="--", linewidth=1)

# spread out dense region near zero
plt.xscale("symlog", linthresh=10)

plt.xlabel("Predicted MMR (deaths per 100,000 live births)")
plt.ylabel("Residual (Actual − Predicted)")
plt.title(f"{best_name} Residuals (symlog scale for readability)")

plt.grid(True, which="both", linewidth=0.4, alpha=0.4)

plt.show()


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

fig, ax = plt.subplots(figsize=(8,6))

ax.scatter(y_pred, residuals, s=18, alpha=0.25, edgecolors="none")
ax.axhline(0, linestyle="--")

ax.set_xlabel("Predicted MMR (deaths per 100,000 live births)")
ax.set_ylabel("Residual (Actual − Predicted)")
ax.set_title(f"{best_name} Residuals")

# inset zoom
axins = inset_axes(ax, width="45%", height="45%", loc="upper right")
axins.scatter(y_pred, residuals, s=12, alpha=0.25)
axins.axhline(0, linestyle="--")

axins.set_xlim(0,250)
axins.set_ylim(-200,200)

mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4")

plt.show()


## Partial dependence and permutation importance


In [ ]:
#partial dependence plot to show how individual features affect predicted MMR
PartialDependenceDisplay.from_estimator(
    best_model, X_train, features=["fertility", "gdp_pc", "pm25"], grid_resolution=30
)
plt.show()


In [ ]:
#fertility graph- strong upward relationship, higher fertility means higher predicted MMR
#gdp graph- negative relationship, higher gdp means lower predicted mmr
#pm2.5- positive nonlinear relationship, higher air pollution means higher risk of MMR


In [ ]:
#permutation feature importance
#permutation importance measures how much model performance decreases when a feature is randomly shuffled

from sklearn.inspection import permutation_importance

result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
perm_imp = pd.Series(result.importances_mean, index=features).sort_values()

perm_imp.plot(kind="barh", title=f"Permutation Importance ({best_name})")
plt.show()
